# Tahap 07 — Harmonisasi Domain-Adversarial (DANN) di Colab GPU

Menguji apakah harmonisasi *device-invariant* yang **dipelajari** (domain-adversarial)
mengalahkan pooling naif (Tahap 05) & harmonisasi-fixed+ccDA (Tahap 06).
Membandingkan **SINGLE / POOL / DANN** dengan backbone **EEGNet** dan **Mamba**.

> **Sumber Tabel 7 manuskrip characterization** ("transfer is model-class-dependent").
> Pakai **`--k 5`** agar konsisten dengan naskah (Tabel 7 + Threats-to-validity menyebut k=5).
> Hasil disimpan ke `07_Harmonization/results/`, disalin ke `manuscripts/characterization/checkpoints/`,
> dan **di-commit** agar tidak hilang lagi.

**Prasyarat:** Runtime GPU; PAT GitHub; Drive `MyDrive/agentic-eeg/derivatives.zip` (epoch mentah,
berisi **KEDUA** dataset: `derivatives/COG-BCI/...` dan `derivatives/STEW/...`).

## 1. Cek GPU

In [ ]:
!nvidia-smi -L
import torch; print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > GPU'

## 2. Mount Drive + clone + install + unzip derivatives

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, getpass, subprocess
DRIVE='/content/drive/MyDrive/agentic-eeg'; REPO='/content/agentic-eeg-decoder'
if not os.path.exists(REPO):
    pat=getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone',f'https://{pat}@github.com/AgentiAI-neural-decoding/agentic-eeg-decoder.git',REPO],check=True); del pat
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
subprocess.run('pip install -q mne mne-icalabel pyriemann onnxruntime huggingface_hub einops "pandas<3"',shell=True)

DERIV=f'{REPO}/02_Preprocessing/derivatives'
import glob
def nfif(ds): return len(glob.glob(f'{DERIV}/{ds}/**/*_proc-clean_epo.fif', recursive=True))
# Unzip hanya dataset yang belum ada (mendukung satu zip gabungan ATAU zip per-dataset di Drive)
for ds in ('COG-BCI','STEW'):
    if nfif(ds): continue
    for z in (f'{DRIVE}/derivatives.zip', f'{DRIVE}/derivatives_{ds.lower().replace("-","")}.zip', f'{DRIVE}/{ds}_derivatives.zip'):
        if os.path.exists(z):
            print(f'unzip {os.path.basename(z)} -> {ds} (~beberapa menit)...')
            subprocess.run(f'unzip -q -o "{z}" -d {REPO}/02_Preprocessing/', shell=True)
            if nfif(ds): break
print('SIAP'); os.chdir(REPO)

## 2b. Verifikasi data SEBELUM training  *(mencegah error 'need at least one array to concatenate')*

Loader DANN mencari `*_proc-clean_epo.fif` di `02_Preprocessing/derivatives/{COG-BCI,STEW}`.
Bila **STEW** atau **COG-BCI** kosong (mis. `derivatives.zip` di Drive tidak memuat keduanya),
training gagal di fold pertama. Sel ini **gagal-cepat dengan pesan jelas** bila ada yang kurang.

In [ ]:
import glob, os
DERIV='02_Preprocessing/derivatives'
counts={ds: len(glob.glob(f'{DERIV}/{ds}/**/*_proc-clean_epo.fif', recursive=True)) for ds in ('COG-BCI','STEW')}
for ds,n in counts.items():
    print(f'  {ds:8s}: {n} file *_proc-clean_epo.fif  {"OK" if n else "<-- KOSONG!"}')
    if n:
        ex = glob.glob(f'{DERIV}/{ds}/**/*_proc-clean_epo.fif', recursive=True)[0]
        print(f'             contoh: {ex}')
missing=[ds for ds,n in counts.items() if n==0]
assert not missing, (
    f'Data {missing} KOSONG. Periksa isi derivatives.zip di Drive:\n'
    f'  zip harus memuat folder derivatives/COG-BCI/... DAN derivatives/STEW/...\n'
    f'  cek cepat:  !unzip -l "{os.path.expanduser("~")}/..."  atau di Drive: !unzip -l "/content/drive/MyDrive/agentic-eeg/derivatives.zip" | grep -E "COG-BCI|STEW" | head\n'
    f'  STEW yang benar berstruktur: derivatives/STEW/sub-XX/sub-XX_task-lo_proc-clean_epo.fif')
print('\nData lengkap -> aman lanjut training.')

## 3. Parameter via flag CLI

Tak mengedit file config (mencegah konflik `git pull`). **`--k 5`** menyelaraskan dengan
naskah (Tabel 7 dilaporkan pada k=5). Ubah `FLAGS` hanya bila sengaja ingin konfigurasi lain.

In [ ]:
FLAGS = "--epochs 60 --cap 6000 --k 5 --lambdas 0.1,0.3,1.0"
print("FLAGS:", FLAGS)

## 4. Backbone EEGNet (CNN) — kedua arah  *(sumber Tabel 7)*

In [ ]:
!python 07_Harmonization/scripts/dann.py --config 07_Harmonization/config/dann.json --backbone cnn --both {FLAGS}

## 5. Backbone Mamba *(opsional — sisi efisiensi/edge, tidak masuk Tabel 7)*

Coba install kernel CUDA `mamba-ssm` (**`--no-build-isolation`** wajib — kalau tidak, build wheel gagal karena env terisolasi tak melihat torch). **Jika tetap gagal, tak apa**: `dann.py` otomatis fallback ke Mamba pure-PyTorch (`mamba(min)`, lebih lambat tapi jalan).

In [ ]:
!pip install -q causal-conv1d --no-build-isolation || echo 'causal-conv1d gagal (lanjut)'
!pip install -q mamba-ssm --no-build-isolation || echo 'mamba-ssm gagal -> fallback mamba(min)'
!python 07_Harmonization/scripts/dann.py --config 07_Harmonization/config/dann.json --backbone mamba --both {FLAGS}

## 6. Verifikasi hasil terhadap Tabel 7 manuskrip

Cetak hasil EEGNet dan bandingkan dengan angka yang dilaporkan di naskah characterization
(Tabel 7: STEW 0.729/0.730/0.750; COG-BCI 0.651/0.718/0.722; teks: COG pool +0.067, DANN≈pool n.s.).
Deep-net stokastik — selisih kecil wajar; bila selisih besar/arah berbeda, perbarui tabel ke hasil ini.

In [ ]:
import json
res = json.load(open('07_Harmonization/results/dann_cnn_results.json'))
# Angka naskah Tabel 7 (best-λ λ=0.1) untuk pembanding cepat:
PAPER = {('STEW','COG-BCI'): (0.729, 0.730, 0.750),
         ('COG-BCI','STEW'): (0.651, 0.718, 0.722)}
print(f"{'target<-source':18s} {'single':>7} {'pool':>7} {'DANN':>7} {'gain_pool':>10} {'DANNvsPOOL':>11} {'bestλ':>6}")
for r in res:
    key = (r['target'], r['source'])
    print(f"{r['target']+'<-'+r['source']:18s} {r['single']:7.3f} {r['pool']:7.3f} {r['best_dann']:7.3f} "
          f"{r['gain_pool']:+10.3f} {r['best_dann_vs_pool']:+11.3f} {r['best_dann_lambda']:>6}")
    if key in PAPER:
        s,p,d = PAPER[key]
        print(f"{'  paper Tabel 7':18s} {s:7.3f} {p:7.3f} {d:7.3f}   Δsingle={r['single']-s:+.3f} Δpool={r['pool']-p:+.3f} Δdann={r['best_dann']-d:+.3f}")
print(f"\nparams={res[0]['params']:,}  latency={res[0]['latency_ms_per_epoch']}ms/epoch  device={res[0]['device']}  n_splits={res[0]['n_splits']}")

## 7. Arsipkan hasil: Drive + `checkpoints/` repo, lalu commit & push

Menyimpan JSON ke (a) Drive dan (b) `manuscripts/characterization/checkpoints/` di dalam repo,
lalu commit & push — supaya artefak Tabel 7 **tersimpan di git** dan tidak hilang lagi.

In [ ]:
import os, glob, shutil, subprocess
DRIVE_CKPT='/content/drive/MyDrive/agentic-eeg/checkpoints'
REPO_CKPT='manuscripts/characterization/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True); os.makedirs(REPO_CKPT, exist_ok=True)
produced = sorted(glob.glob('07_Harmonization/results/*.json'))
for f in produced:
    shutil.copy2(f, DRIVE_CKPT)      # ke Drive (backup)
    shutil.copy2(f, REPO_CKPT)       # ke repo (akan di-commit)
    print('arsip:', os.path.basename(f))

# Commit & push (hasil + notebook ini dengan output-nya)
subprocess.run('git config user.email "mubaroq@digitalbdg.ac.id"', shell=True)
subprocess.run('git config user.name "mubaroq"', shell=True)
subprocess.run(f'git add {REPO_CKPT}/dann_*.json 07_Harmonization/colab_dann.ipynb', shell=True)
subprocess.run('git commit -m "Stage07: DANN k=5 final results (source of characterization Table 7)"', shell=True)
print(subprocess.run('git push', shell=True, capture_output=True, text=True).stderr[-500:])

## Interpretasi
- **DANN vs POOL**: apakah adversarial harmonization mengurangi negative transfer pooling?
- **DANN vs SINGLE**: apakah transfer jadi positif?
- **EEGNet vs Mamba**: akurasi setara? params/latency Mamba (efisiensi/edge)?
- Hasil negatif pun informatif (menegaskan domain shift sbg tembok keras).
- Setelah commit (sel 7), `git pull` di lokal untuk menarik `dann_cnn_results.json` ke `checkpoints/`,
  lalu lakukan cross-check final Tabel 7 sebelum submit.